# Grafana & Time-Series Database · Edge-Computing Notebook

In the previous lab you collected telemetry. Now you will **store** it in a time-series database and **visualize** it with Grafana — the storage and dashboard layer that runs on the edge node.

```text
Fake Sensor -> Time-Series Database (InfluxDB) -> Queries -> Grafana Dashboard
```

The fake sensor generates readings like:

```json
{
  "temperature": 34.2,
  "humidity": 41.8,
  "motion": true,
  "battery": 76
}
```

The time-series database stores each reading with a timestamp so you can query changes over time, and Grafana turns those queries into live charts.

Example questions this stack can answer:

```text
What was the latest sensor value?
What was the average temperature over the last 5 minutes?
When did the device stop sending data?
How much data should be retained?
```

**You will:**
- run InfluxDB in a container and write sensor readings using line protocol
- query recent values, latest values, and time-window aggregates with Flux
- run a full sensor → InfluxDB → Grafana pipeline with Docker Compose
- build a live Grafana dashboard and watch it show a gap when a sensor goes silent
- create a retention bucket and monitor how fast the database grows on disk

## How this notebook works · cells and a Jupyter terminal

Some steps run once and return (starting a container, a `curl` write, a query) — those are ordinary notebook cells. Other steps run live and never return on their own: watching the Compose pipeline's logs scroll, or following the sensor writer's output. For those, this notebook **writes a small script into your lab folder and asks you to run it in a Jupyter terminal.**

Two labels are used throughout:

- **[Notebook cell]** · run the cell right here.
- **[Terminal]** · open a Jupyter terminal — **File ▸ New ▸ Terminal** (or the Terminal tile in the Launcher) — and run the given command. The terminal is a real shell **on the DGX Spark**, the same machine as the Docker daemon.

### One config, everywhere

Terminal shells do not share the notebook kernel's variables. So the setup cell below writes `~/grafanaTSDBLab/labEnv.sh`, and every helper script starts with `source ~/grafanaTSDBLab/labEnv.sh`. Your ports, org, bucket, and token are derived once and agree everywhere — notebook cells, terminals, and Docker Compose.

**Requirements:** run this notebook with a Python 3 kernel on the DGX Spark, where Docker is available to your user.

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, preflight checks, and checkpoints.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

---
## Part 0 · Before You Begin

You are working on a shared **DGX Spark** edge device, through JupyterLab running on the DGX Spark itself. **Every command in this notebook runs on the DGX Spark**, not on your own computer.

Because the DGX Spark is shared:

- use your own home folder and keep your files under it
- prefix Docker resources and other shared names with your account name using `${USER}`
- use only the ports assigned to you (derived automatically below)

> Everything here runs in a **Jupyter Terminal** (**File ▸ New ▸ Terminal**) - every helper script this notebook writes runs the same way there. (Students reach the class box through JupyterHub, not SSH.)

---
## Part 1 · Set Your Assigned Ports

Docker resources run through the same Docker daemon on the DGX Spark, so host ports are shared across the whole device. Each student or group must use two **unique** ports: one for InfluxDB, one for Grafana.

**[Notebook cell]** Your ports are derived automatically from the digits in your username (`student07` → InfluxDB `8007`, Grafana `3007`), so nobody on the shared device collides. If your instructor assigned specific ports instead, pass `portOverrides={"INFLUX_PORT": ..., "GRAFANA_PORT": ...}`. The cell also sets the database and dashboard settings for this lab, creates `~/grafanaTSDBLab/`, and writes `labEnv.sh` for the terminal scripts.

In [ ]:
import os

labConfig = setupLab(
    "grafanaTSDBLab",
    ports={"INFLUX_PORT": 8000, "GRAFANA_PORT": 3000},
    extraEnv={
        "INFLUX_ORG": "edge-lab",
        "INFLUX_BUCKET": "sensor-data",
        "INFLUX_USER": f"{os.environ.get('USER', 'student01')}-influx",
        "INFLUX_PASSWORD": "edgepass12345",
        "INFLUX_TOKEN": f"{os.environ.get('USER', 'student01')}-influx-token-12345",
        "GRAFANA_PASSWORD": "edgegrafana12345",
    },
)

> These values are exported to the notebook environment **and** written to `~/grafanaTSDBLab/labEnv.sh`. Unlike a plain `export` in one SSH shell — which dies with the session — any terminal script that sources `labEnv.sh` sees exactly the same values.

### Preflight · check your environment

Run this once at the start. It confirms Docker, Compose, and `curl` work and that your assigned InfluxDB port is not already taken by another student. Fix any failing check before continuing.

In [ ]:
import os

def portFree(envName):
    # Passes while nothing is listening on the port yet (your own InfluxDB from
    # Part 3 counts too, so this check matters only before you start it).
    def probe():
        portValue = int(os.environ.get(envName, "0"))
        listening, _ = portListening(portValue)()
        detail = (f"port {portValue} already in use - fine if it is YOUR container, "
                  "otherwise ask for another port") if listening else f"port {portValue} free"
        return (not listening), detail
    return probe

preflight(
    [
        check("docker daemon", dockerDaemonUp(),
              hint="Docker must be running and your account must have an instructor-provisioned rootless Podman socket - ask the instructor.",
              link="https://docs.docker.com/engine/daemon/troubleshoot/"),
        check("docker compose", composeAvailable(),
              hint="Compose v2 ships with Docker Engine as the `docker compose` plugin.",
              link="https://docs.docker.com/compose/install/"),
        check("curl on PATH", commandOnPath("curl"),
              hint="Install curl (`sudo apt install curl`) or ask the instructor - this lab drives InfluxDB with it.",
              link="https://curl.se/docs/manpage.html"),
        check("INFLUX_PORT not taken", portFree("INFLUX_PORT"),
              hint="Another process is on your InfluxDB port. If it is your own container from a previous run, "
                   "continue; otherwise re-run setupLab with portOverrides for a free port.",
              link="https://docs.docker.com/engine/network/#published-ports"),
        check("GRAFANA_PORT not taken", portFree("GRAFANA_PORT"),
              hint="Another process is on your Grafana port. If it is your own stack from a previous run, "
                   "continue; otherwise re-run setupLab with portOverrides for a free port.",
              link="https://docs.docker.com/engine/network/#published-ports"),
    ],
    infoRows=[
        ("your USER", os.environ.get("USER", "?")),
        ("your INFLUX_PORT", os.environ.get("INFLUX_PORT", "?")),
        ("your GRAFANA_PORT", os.environ.get("GRAFANA_PORT", "?")),
    ],
)

In [ ]:
%cd ~/grafanaTSDBLab

---
## Part 2 · Verify Docker and Basic Tools

**[Notebook cell]** Check the tool versions, then look at what is already running on the shared daemon. If another student's container is publishing your assigned port, ask the instructor for a different port before continuing.

In [ ]:
dockerVersions(runHelloWorld=False)
!curl --version | head -n 1
dockerPs()

Time-series databases (used for sensor readings, server/device metrics, logs, and IoT data) are optimized for many *timestamped events*, unlike a normal table that mainly stores current state.

---
## Part 3 · Run an InfluxDB Time-Series Database

**[Notebook cell]** Create a Docker network for this lab and a volume for database storage:

In [ ]:
!docker network create ${USER}-tsdb-net 2>/dev/null || true
!docker volume inspect ${USER}-influxdbData >/dev/null 2>&1 || docker volume create ${USER}-influxdbData

**[Notebook cell]** Start InfluxDB. The `docker rm -f` line removes any earlier copy first, so the cell is safe to re-run:

In [ ]:
!docker rm -f ${USER}-influxdb 2>/dev/null || true
!docker run -d \
  --name ${USER}-influxdb \
  --network ${USER}-tsdb-net \
  -p $INFLUX_PORT:8086 \
  -v ${USER}-influxdbData:/var/lib/influxdb2 \
  -e DOCKER_INFLUXDB_INIT_MODE=setup \
  -e DOCKER_INFLUXDB_INIT_USERNAME=$INFLUX_USER \
  -e DOCKER_INFLUXDB_INIT_PASSWORD=$INFLUX_PASSWORD \
  -e DOCKER_INFLUXDB_INIT_ORG=$INFLUX_ORG \
  -e DOCKER_INFLUXDB_INIT_BUCKET=$INFLUX_BUCKET \
  -e DOCKER_INFLUXDB_INIT_ADMIN_TOKEN=$INFLUX_TOKEN \
  influxdb:2.7

**[Notebook cell]** Check that the container is running and healthy:

In [ ]:
import os
import time

time.sleep(3)   # give the database a moment to initialize
dockerPs(namePattern="influxdb")
dockerLogs(f"{os.environ.get('USER', 'student01')}-influxdb", tail=10)

In [ ]:
!curl -s http://$DEVICE_ADDR:$INFLUX_PORT/health && echo

**You should see:** a `{"status":"pass"}` response — the database is up. **See it live** — run the next cell for an inline snapshot and an **Open interactively** link that opens the service in your browser, proxied through JupyterHub: (log in with your `INFLUX_USER` / `INFLUX_PASSWORD`), but the rest of this exercise drives the database with `curl` and Compose, so the UI is not required.

This creates a local time-series database that can store sensor readings.

### Checkpoint · Part 3

In [ ]:
import os
showDashboard(port=os.environ.get("INFLUX_PORT"), label="InfluxDB UI")

In [ ]:
import os

userName = os.environ.get("USER", "student01")
influxPort = os.environ.get("INFLUX_PORT", "8086")

checkpoint(
    "Part 3 - InfluxDB running",
    [
        check("influxdb container running", containerRunning(f"{userName}-influxdb"),
              hint="Re-run the two Part 3 cells above, then look at the container logs with "
                   "dockerLogs(f\"{userName}-influxdb\") for the failure reason.",
              link="https://docs.influxdata.com/influxdb/v2/install/?t=Docker"),
        check("storage volume exists", volumeExists(f"{userName}-influxdbData"),
              hint="Re-run the `docker volume create` cell in Part 3.",
              link="https://docs.docker.com/engine/storage/volumes/"),
        check("health endpoint answers 'pass'",
              httpOk(f"http://{deviceAddress()}:{influxPort}/health", expectText="pass"),
              hint="The container may still be starting - wait a few seconds and re-run this checkpoint. "
                   "If it keeps failing, check `dockerLogs` above for an init error.",
              link="https://docs.influxdata.com/influxdb/v2/api/#operation/GetHealth"),
    ],
    successNote="Your time-series database is up, with its storage on a named volume.",
    docLink="https://docs.influxdata.com/influxdb/v2/",
)

---
## Part 4 · Write a Single Sensor Reading

InfluxDB accepts data using line protocol, which has this structure:

```text
measurement,tag_key=tag_value field_key=field_value timestamp
```

Example:

```text
environment,device=sensor01,room=lab temperature=24.7,humidity=41.2,motion=false,battery=93i
```

Here `environment` is the measurement, `device`/`room` are tags, `temperature`/`humidity`/etc. are fields, and the omitted timestamp means InfluxDB uses server time (the `New term` box below recaps this).

**[Notebook cell]** Write one point:

In [ ]:
!curl --request POST \
  "http://$DEVICE_ADDR:$INFLUX_PORT/api/v2/write?org=$INFLUX_ORG&bucket=$INFLUX_BUCKET&precision=s" \
  --header "Authorization: Token $INFLUX_TOKEN" \
  --header "Content-Type: text/plain; charset=utf-8" \
  --data-binary "environment,device=sensor01,room=lab temperature=24.7,humidity=41.2,motion=false,battery=93i"

**You should see:** nothing at all. A successful write returns an empty `204 No Content` response, so no output *is* the success signal. If you instead get a JSON error mentioning `unauthorized` or `bucket not found`, re-run the Part 1 setup cell so `INFLUX_TOKEN`, `INFLUX_ORG`, and `INFLUX_BUCKET` are exported.

> **New term · line protocol:** InfluxDB's compact text format for one reading: `measurement,tag=value field=value`. Tags are indexed labels (like `device`); fields are the numbers you measure (the trailing `i` marks an integer).

**[Notebook cell]** Write another point with a current Unix timestamp:

In [ ]:
!NOW=$(date +%s); curl --request POST \
  "http://$DEVICE_ADDR:$INFLUX_PORT/api/v2/write?org=$INFLUX_ORG&bucket=$INFLUX_BUCKET&precision=s" \
  --header "Authorization: Token $INFLUX_TOKEN" \
  --header "Content-Type: text/plain; charset=utf-8" \
  --data-binary "environment,device=sensor01,room=lab temperature=25.2,humidity=40.8,motion=true,battery=92i $NOW"

This shows that time-series data is a value attached to time and metadata, not just a value.

> **Optional · ingest your Lab 3 telemetry:** the readings you saved to `~/telemetryLab/data/telemetry.jsonl` map directly onto this `environment` measurement. If you want real data instead of the fake stream, loop over that file and POST each line the same way as above (preserving each reading's original timestamp). The fake sensor writer in Part 6 generates fresh data either way, so this is not required to continue.

### Checkpoint · Part 4

In [ ]:
import os

influxPort = os.environ.get("INFLUX_PORT", "8086")
influxOrg = os.environ.get("INFLUX_ORG", "edge-lab")
influxToken = os.environ.get("INFLUX_TOKEN", "")
recentQuery = (
    f'curl -s --request POST "http://{deviceAddress()}:{influxPort}/api/v2/query?org={influxOrg}" '
    f'--header "Authorization: Token {influxToken}" '
    '--header "Accept: application/csv" --header "Content-type: application/vnd.flux" '
    '--data \'from(bucket:"sensor-data") |> range(start: -1h) '
    '|> filter(fn: (r) => r._measurement == "environment")\''
)

checkpoint(
    "Part 4 - first points written",
    [
        check("write API reachable", httpOk(f"http://{deviceAddress()}:{influxPort}/health", expectText="pass"),
              hint="InfluxDB is not answering - re-run the Part 3 cells to start the container.",
              link="https://docs.influxdata.com/influxdb/v2/api/#operation/GetHealth"),
        check("environment points stored", outputContains(recentQuery, "environment"),
              hint="No stored points came back. Re-run the two curl write cells above. An error mentioning "
                   "'unauthorized' or 'bucket not found' means the INFLUX_* values are wrong - re-run the "
                   "Part 1 setup cell, then write again.",
              link="https://docs.influxdata.com/influxdb/v2/write-data/developer-tools/api/"),
    ],
    successNote="Two timestamped points are stored - a reading is now a value attached to time and metadata.",
    docLink="https://docs.influxdata.com/influxdb/v2/reference/syntax/line-protocol/",
)

---
## Part 5 · Query Recent Readings

**[Notebook cell]** Create a folder for query files, then write the two Flux queries (the original lab creates these with `nano`; here `%%writefile` does the same job):

In [ ]:
%cd ~/grafanaTSDBLab
!mkdir -p queries

In [ ]:
%%writefile queries/recentReadings.flux
from(bucket:"sensor-data")
  |> range(start: -15m)
  |> filter(fn: (r) => r._measurement == "environment")

In [ ]:
%%writefile queries/latestReading.flux
from(bucket:"sensor-data")
  |> range(start: -15m)
  |> filter(fn: (r) => r._measurement == "environment")
  |> last()

In [ ]:
# Preview the query files we just wrote.
showFile("~/grafanaTSDBLab/queries/recentReadings.flux")
showFile("~/grafanaTSDBLab/queries/latestReading.flux")

**[Notebook cell]** Run the recent-readings query:

In [ ]:
!curl --request POST \
  "http://$DEVICE_ADDR:$INFLUX_PORT/api/v2/query?org=$INFLUX_ORG" \
  --header "Authorization: Token $INFLUX_TOKEN" \
  --header "Accept: application/csv" \
  --header "Content-type: application/vnd.flux" \
  --data-binary @queries/recentReadings.flux

**[Notebook cell]** Run the latest-reading query:

In [ ]:
!curl --request POST \
  "http://$DEVICE_ADDR:$INFLUX_PORT/api/v2/query?org=$INFLUX_ORG" \
  --header "Authorization: Token $INFLUX_TOKEN" \
  --header "Accept: application/csv" \
  --header "Content-type: application/vnd.flux" \
  --data-binary @queries/latestReading.flux

Each field is returned as its own time-series value.

### Checkpoint · Part 5

In [ ]:
import os

influxPort = os.environ.get("INFLUX_PORT", "8086")
influxOrg = os.environ.get("INFLUX_ORG", "edge-lab")
influxToken = os.environ.get("INFLUX_TOKEN", "")
latestFluxPath = os.path.expanduser("~/grafanaTSDBLab/queries/latestReading.flux")
latestQuery = (
    f'curl -s --request POST "http://{deviceAddress()}:{influxPort}/api/v2/query?org={influxOrg}" '
    f'--header "Authorization: Token {influxToken}" '
    '--header "Accept: application/csv" --header "Content-type: application/vnd.flux" '
    f'--data-binary @{latestFluxPath}'
)

checkpoint(
    "Part 5 - Flux queries",
    [
        check("recentReadings.flux exists", fileExists("~/grafanaTSDBLab/queries/recentReadings.flux"),
              hint="Re-run the %%writefile cell for queries/recentReadings.flux above.",
              link="https://docs.influxdata.com/influxdb/v2/query-data/flux/"),
        check("latestReading.flux exists", fileExists("~/grafanaTSDBLab/queries/latestReading.flux"),
              hint="Re-run the %%writefile cell for queries/latestReading.flux above.",
              link="https://docs.influxdata.com/influxdb/v2/query-data/flux/"),
        check("latest-reading query returns rows", outputContains(latestQuery, "environment"),
              hint="If your writes in Part 4 were more than 15 minutes ago they fall outside range(start: -15m) - "
                   "re-run the Part 4 write cells, then this checkpoint.",
              link="https://docs.influxdata.com/influxdb/v2/api/#operation/PostQuery"),
    ],
    successNote="You can ask the database for recent and latest values - the building blocks of every dashboard panel.",
    docLink="https://docs.influxdata.com/influxdb/v2/query-data/flux/",
)

---
## Part 6 · Build a Fake Sensor Writer

A dashboard needs a steady stream of data. Build a sensor that writes a new reading every two seconds.

**[Notebook cell]** Create the folder and write the source files:

In [ ]:
!mkdir -p ~/grafanaTSDBLab/sensorWriter
%cd ~/grafanaTSDBLab/sensorWriter

In [ ]:
%%writefile sensorWriter.py
import os
import random
import time
import requests

influxURL = os.environ["INFLUX_URL"]
influxOrg = os.environ["INFLUX_ORG"]
influxBucket = os.environ["INFLUX_BUCKET"]
influxToken = os.environ["INFLUX_TOKEN"]
deviceID = os.environ.get("DEVICE_ID", "sensor01")
room = os.environ.get("ROOM", "lab")

writeURL = f"{influxURL}/api/v2/write"

headers = {
    "Authorization": f"Token {influxToken}",
    "Content-Type": "text/plain; charset=utf-8",
}

params = {
    "org": influxOrg,
    "bucket": influxBucket,
    "precision": "s",
}

while True:
    temperature = round(random.uniform(22, 40), 2)
    humidity = round(random.uniform(35, 60), 2)
    motion = random.choice(["true", "false"])
    battery = random.randint(20, 100)

    line = (
        f"environment,device={deviceID},room={room} "
        f"temperature={temperature},humidity={humidity},motion={motion},battery={battery}i"
    )

    try:
        response = requests.post(
            writeURL,
            params=params,
            headers=headers,
            data=line,
            timeout=3,
        )
        print(f"sent: {line} | status: {response.status_code}", flush=True)
    except Exception as writeError:
        print(f"failed to write reading: {writeError}", flush=True)

    time.sleep(2)

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

RUN pip install requests

COPY sensorWriter.py .

CMD ["python", "sensorWriter.py"]

In [ ]:
# Preview the two files we just wrote.
showFile("~/grafanaTSDBLab/sensorWriter/sensorWriter.py")
showFile("~/grafanaTSDBLab/sensorWriter/Dockerfile")

**[Notebook cell]** Build the image:

> **Naming reminder:** the image tag stays lowercase (`${USER}-sensor-writer`) while the container it runs is camelCase (`${USER}-sensorWriter`). Docker requires image names to be lowercase; container names may use capitals. See Lab 02, Part 13 for the full explanation.

In [ ]:
!docker build -t ${USER}-sensor-writer .

The original lab runs the writer in the foreground with `--rm` and stops it with Ctrl-C. A notebook cell would hang on that, so here we start it **detached** (`-d`) and watch its logs — same container, same behavior:

In [ ]:
!docker rm -f ${USER}-sensorWriter 2>/dev/null || true
!docker run -d \
  --name ${USER}-sensorWriter \
  --network ${USER}-tsdb-net \
  -e INFLUX_URL=http://${USER}-influxdb:8086 \
  -e INFLUX_ORG=$INFLUX_ORG \
  -e INFLUX_BUCKET=$INFLUX_BUCKET \
  -e INFLUX_TOKEN=$INFLUX_TOKEN \
  -e DEVICE_ID=sensor01 \
  -e ROOM=lab \
  ${USER}-sensor-writer

In [ ]:
import os
import time

time.sleep(5)   # let a couple of readings go out first
dockerLogs(f"{os.environ.get('USER', 'student01')}-sensorWriter", tail=5)

In [ ]:
%%writefile watchWriter.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/grafanaTSDBLab/labEnv.sh

echo "Live sensorWriter output - press Ctrl-C to stop watching (the container keeps running)."
docker logs -f "${USER}-sensorWriter"

In [ ]:
!chmod +x watchWriter.sh

**[Terminal]** To watch the stream live — one `sent: ...` line every 2 seconds — run:

```bash
~/grafanaTSDBLab/sensorWriter/watchWriter.sh
```

**[Notebook cell]** While it streams, query the latest readings (the original lab does this from a second SSH session — the notebook plays that role here):

In [ ]:
!curl --request POST \
  "http://$DEVICE_ADDR:$INFLUX_PORT/api/v2/query?org=$INFLUX_ORG" \
  --header "Authorization: Token $INFLUX_TOKEN" \
  --header "Accept: application/csv" \
  --header "Content-type: application/vnd.flux" \
  --data-binary @$HOME/grafanaTSDBLab/queries/latestReading.flux

The writer keeps running in the background, adding a new time-stamped reading every 2 seconds. Part 8 will replace it (and the standalone InfluxDB) with the Compose version.

### Checkpoint · Part 6

In [ ]:
import os

userName = os.environ.get("USER", "student01")
influxPort = os.environ.get("INFLUX_PORT", "8086")
influxOrg = os.environ.get("INFLUX_ORG", "edge-lab")
influxToken = os.environ.get("INFLUX_TOKEN", "")
recentQuery = (
    f'curl -s --request POST "http://{deviceAddress()}:{influxPort}/api/v2/query?org={influxOrg}" '
    f'--header "Authorization: Token {influxToken}" '
    '--header "Accept: application/csv" --header "Content-type: application/vnd.flux" '
    '--data \'from(bucket:"sensor-data") |> range(start: -5m) '
    '|> filter(fn: (r) => r._measurement == "environment") |> last()\''
)

checkpoint(
    "Part 6 - fake sensor writer streaming",
    [
        check("writer image built", imageExists(f"{userName}-sensor-writer"),
              hint="Re-run the `docker build` cell in Part 6.",
              link="https://docs.docker.com/reference/cli/docker/buildx/build/"),
        check("writer container running", containerRunning(f"{userName}-sensorWriter"),
              hint="Re-run the detached `docker run -d` cell in Part 6, then check its logs with "
                   "watchWriter.sh or dockerLogs(...).",
              link="https://docs.docker.com/reference/cli/docker/container/run/"),
        check("fresh sensor01 data in the last 5 minutes", outputContains(recentQuery, "sensor01"),
              hint="The writer may be failing to reach InfluxDB - check its logs. Both containers must share "
                   "the ${USER}-tsdb-net network (Part 3 and Part 6 run cells).",
              link="https://docs.docker.com/engine/network/"),
    ],
    successNote="A containerized sensor is streaming timestamped readings into your database every 2 seconds.",
    docLink="https://docs.influxdata.com/influxdb/v2/write-data/",
)

---
## Part 7 · Query Aggregates Over Time

Time-series databases can aggregate values over time windows.

**[Notebook cell]** Create `queries/avgTemperature.flux` and run it:

In [ ]:
%cd ~/grafanaTSDBLab

In [ ]:
%%writefile queries/avgTemperature.flux
from(bucket:"sensor-data")
  |> range(start: -10m)
  |> filter(fn: (r) => r._measurement == "environment")
  |> filter(fn: (r) => r._field == "temperature")
  |> aggregateWindow(every: 1m, fn: mean, createEmpty: false)

In [ ]:
!curl --request POST \
  "http://$DEVICE_ADDR:$INFLUX_PORT/api/v2/query?org=$INFLUX_ORG" \
  --header "Authorization: Token $INFLUX_TOKEN" \
  --header "Accept: application/csv" \
  --header "Content-type: application/vnd.flux" \
  --data-binary @queries/avgTemperature.flux

This is the difference between storing raw events and asking questions about behavior over time. In a moment you will run the same kind of query inside Grafana.

### Checkpoint · Part 7

In [ ]:
import os

influxPort = os.environ.get("INFLUX_PORT", "8086")
influxOrg = os.environ.get("INFLUX_ORG", "edge-lab")
influxToken = os.environ.get("INFLUX_TOKEN", "")
avgFluxPath = os.path.expanduser("~/grafanaTSDBLab/queries/avgTemperature.flux")
avgQuery = (
    f'curl -s --request POST "http://{deviceAddress()}:{influxPort}/api/v2/query?org={influxOrg}" '
    f'--header "Authorization: Token {influxToken}" '
    '--header "Accept: application/csv" --header "Content-type: application/vnd.flux" '
    f'--data-binary @{avgFluxPath}'
)

checkpoint(
    "Part 7 - aggregate query",
    [
        check("avgTemperature.flux exists", fileExists("~/grafanaTSDBLab/queries/avgTemperature.flux"),
              hint="Re-run the %%writefile cell for queries/avgTemperature.flux above.",
              link="https://docs.influxdata.com/flux/v0/stdlib/universe/aggregatewindow/"),
        check("windowed means come back", outputContains(avgQuery, "temperature"),
              hint="Empty results usually mean no data in the last 10 minutes - make sure the sensor writer "
                   "from Part 6 is still running, then re-run this checkpoint.",
              link="https://docs.influxdata.com/influxdb/v2/query-data/flux/window-aggregate/"),
    ],
    successNote="You aggregated raw events into per-minute means - exactly what Grafana panels do continuously.",
    docLink="https://docs.influxdata.com/influxdb/v2/query-data/flux/window-aggregate/",
)

---
## Part 8 · Multi-Container Pipeline with Grafana

Now make the system realistic and add Grafana as the dashboard layer.

```text
Sensor Writer Container -> InfluxDB Container -> Grafana Container
```

Target folder structure:

```text
tsdbComposeLab/
  sensor/
    sensorWriter.py
    Dockerfile
  grafana/
    provisioning/
      datasources/
        influxdb.yaml
  compose.yaml
```

**[Notebook cell]** Stop the single containers (Compose reuses the same ports), create the folders, and copy the sensor writer files from the previous part:

In [ ]:
!docker rm -f ${USER}-sensorWriter 2>/dev/null || true
!docker rm -f ${USER}-influxdb 2>/dev/null || true
!mkdir -p ~/grafanaTSDBLab/tsdbComposeLab/sensor
!mkdir -p ~/grafanaTSDBLab/tsdbComposeLab/grafana/provisioning/datasources
%cd ~/grafanaTSDBLab/tsdbComposeLab
!cp ~/grafanaTSDBLab/sensorWriter/sensorWriter.py sensor/
!cp ~/grafanaTSDBLab/sensorWriter/Dockerfile sensor/

### Provision the Grafana data source

Instead of clicking through the Grafana UI to connect it to InfluxDB, provide a provisioning file so the data source is configured automatically on startup.

In [ ]:
%%writefile grafana/provisioning/datasources/influxdb.yaml
apiVersion: 1

datasources:
  - name: InfluxDB
    type: influxdb
    access: proxy
    url: http://influxdb:8086
    jsonData:
      version: Flux
      organization: ${INFLUX_ORG}
      defaultBucket: ${INFLUX_BUCKET}
    secureJsonData:
      token: ${INFLUX_TOKEN}

Grafana expands the `${...}` values from the environment variables passed to its container, which you set in the Compose file below.

### Create `compose.yaml`

In [ ]:
%%writefile compose.yaml
services:
  influxdb:
    image: influxdb:2.7
    ports:
      - "${INFLUX_PORT}:8086"
    environment:
      DOCKER_INFLUXDB_INIT_MODE: setup
      DOCKER_INFLUXDB_INIT_USERNAME: ${INFLUX_USER}
      DOCKER_INFLUXDB_INIT_PASSWORD: ${INFLUX_PASSWORD}
      DOCKER_INFLUXDB_INIT_ORG: ${INFLUX_ORG}
      DOCKER_INFLUXDB_INIT_BUCKET: ${INFLUX_BUCKET}
      DOCKER_INFLUXDB_INIT_ADMIN_TOKEN: ${INFLUX_TOKEN}
    volumes:
      - influxdb-data:/var/lib/influxdb2

  sensor-writer:
    build: ./sensor
    depends_on:
      - influxdb
    environment:
      INFLUX_URL: http://influxdb:8086
      INFLUX_ORG: ${INFLUX_ORG}
      INFLUX_BUCKET: ${INFLUX_BUCKET}
      INFLUX_TOKEN: ${INFLUX_TOKEN}
      DEVICE_ID: sensor01
      ROOM: lab

  grafana:
    image: grafana/grafana
    depends_on:
      - influxdb
    ports:
      - "${GRAFANA_PORT}:3000"
    environment:
      GF_SECURITY_ADMIN_USER: admin
      GF_SECURITY_ADMIN_PASSWORD: ${GRAFANA_PASSWORD}
      INFLUX_ORG: ${INFLUX_ORG}
      INFLUX_BUCKET: ${INFLUX_BUCKET}
      INFLUX_TOKEN: ${INFLUX_TOKEN}
    volumes:
      - ./grafana/provisioning:/etc/grafana/provisioning
      - grafana-data:/var/lib/grafana

volumes:
  influxdb-data:
  grafana-data:

In [ ]:
# Preview the two files we just wrote.
showFile("~/grafanaTSDBLab/tsdbComposeLab/grafana/provisioning/datasources/influxdb.yaml")
showFile("~/grafanaTSDBLab/tsdbComposeLab/compose.yaml")

> `grafana/grafana` pulls the latest Grafana image. Your instructor may ask you to pin a specific version tag; if so, replace `grafana/grafana` with the tag they provide.

The original lab runs `docker compose up --build` in the foreground and watches the logs scroll. A notebook cell would hang on that, so here we start the stack **detached** (`-d`). The helper scripts below give you the live-log experience in a terminal.

**[Notebook cell]** Run the full pipeline using your account name as the Compose project name:

In [ ]:
!docker compose -p ${USER}-tsdb-lab up -d --build

In [ ]:
%%writefile runPipeline.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/grafanaTSDBLab/labEnv.sh
cd ~/grafanaTSDBLab/tsdbComposeLab

echo "Building and starting the pipeline. Live logs below."
echo "Grafana port ${GRAFANA_PORT}, InfluxDB port ${INFLUX_PORT} - open them in your browser via showDashboard(port=...) in the notebook."
echo "Press Ctrl-C to stop, or run stopPipeline.sh from another terminal."
docker compose -p "${USER}-tsdb-lab" up --build

In [ ]:
%%writefile watchPipeline.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/grafanaTSDBLab/labEnv.sh
cd ~/grafanaTSDBLab/tsdbComposeLab

echo "Following pipeline logs - press Ctrl-C to stop watching (the stack keeps running)."
docker compose -p "${USER}-tsdb-lab" logs -f

In [ ]:
%%writefile stopPipeline.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/grafanaTSDBLab/labEnv.sh
cd ~/grafanaTSDBLab/tsdbComposeLab

docker compose -p "${USER}-tsdb-lab" down

In [ ]:
!chmod +x runPipeline.sh watchPipeline.sh stopPipeline.sh

**[Terminal]** Watch the sensorWriter logs scroll live while the stack runs detached:

```bash
~/grafanaTSDBLab/tsdbComposeLab/watchPipeline.sh
```

(Alternatively, stop the detached stack with `stopPipeline.sh` and use `runPipeline.sh` for the classic foreground `up --build`.)

### Why use `-p ${USER}-tsdb-lab`?

Docker Compose creates project-specific containers, networks, images, and volumes. Because students share the same Docker daemon, giving each project a unique name keyed on `${USER}` keeps everyone's Compose resources separate.

**[Notebook cell]** Confirm all three services came up:

In [ ]:
import os
import time

time.sleep(10)   # give the stack a moment to build and start
dockerPs(namePattern="tsdb-lab")
dockerLogs(f"{os.environ.get('USER', 'student01')}-tsdb-lab-sensorWriter-1", tail=5)

### Checkpoint · Part 8

In [ ]:
import os

userName = os.environ.get("USER", "student01")
influxPort = os.environ.get("INFLUX_PORT", "8086")
grafanaPort = os.environ.get("GRAFANA_PORT", "3000")

checkpoint(
    "Part 8 - Compose pipeline up",
    [
        check("influxdb service running", containerRunning(f"{userName}-tsdb-lab-influxdb-1"),
              hint="Re-run the `docker compose up -d --build` cell, then `dockerPs(namePattern='tsdb-lab')` "
                   "to see what failed.",
              link="https://docs.docker.com/compose/"),
        check("sensorWriter service running", containerRunning(f"{userName}-tsdb-lab-sensor-writer-1"),
              hint="If the build failed, check the compose output above - the sensor folder must contain "
                   "sensorWriter.py and the Dockerfile copied earlier in Part 8.",
              link="https://docs.docker.com/compose/how-tos/file-watch/"),
        check("grafana service running", containerRunning(f"{userName}-tsdb-lab-grafana-1"),
              hint="Grafana may still be pulling its image - wait, then re-run this checkpoint. If your "
                   "GRAFANA_PORT is taken, `docker compose up` reports a port bind error above.",
              link="https://grafana.com/docs/grafana/latest/setup-grafana/installation/docker/"),
        check("InfluxDB health endpoint", httpOk(f"http://{deviceAddress()}:{influxPort}/health", expectText="pass"),
              hint="The database may still be initializing - wait a few seconds and re-run this checkpoint.",
              link="https://docs.influxdata.com/influxdb/v2/api/#operation/GetHealth"),
        check("Grafana login page", httpOk(f"http://{deviceAddress()}:{grafanaPort}/login", expectText="Grafana"),
              hint="Grafana can take ~20s on first start. Wait and re-run; if it keeps failing, check "
                   "`dockerLogs(f'{userName}-tsdb-lab-grafana-1')`.",
              link="https://grafana.com/docs/grafana/latest/setup-grafana/"),
    ],
    successNote="The full sensor -> InfluxDB -> Grafana pipeline is running in containers on the edge node.",
    docLink="https://docs.docker.com/compose/",
)

---
## Part 9 · Set Up the Grafana Dashboard

**See it live** — run the next cell for an inline snapshot and an **Open interactively** link that opens the service in your browser, proxied through JupyterHub:


In [ ]:
import os
showDashboard(port=os.environ.get("GRAFANA_PORT"), label="Grafana")

Log in with:

```text
username: admin
password: value of GRAFANA_PASSWORD   (edgegrafana12345 unless you changed it)
```

Confirm the data source loaded from provisioning:

```text
Connections -> Data sources -> InfluxDB
```

Scroll down and click **Save & test**.

**You should see:** a green success message such as `datasource is working`. That confirms Grafana reached InfluxDB inside the Compose network and the token is valid.

> **If it doesn't work:** a red error usually means the token didn't reach Grafana. Confirm the Part 1 setup cell ran *before* the `docker compose up` cell, then recreate the stack (`docker compose -p ${USER}-tsdb-lab up -d --force-recreate grafana` in a terminal from the compose folder). A "no such host" error means you opened the wrong port; use your assigned `GRAFANA_PORT`, not the InfluxDB one.

Now build a panel:

```text
Dashboards -> New -> New dashboard -> Add visualization -> select InfluxDB
```

In the query editor, switch the editor to **code** mode and paste this Flux query:

```text
from(bucket: "sensor-data")
  |> range(start: -15m)
  |> filter(fn: (r) => r._measurement == "environment")
  |> filter(fn: (r) => r._field == "temperature")
```

Click **Run query**. A live temperature line should appear, updating as the sensor writer adds data. Set the dashboard auto-refresh (top right) to **5s** and the time range to **Last 15 minutes**.

Add a second panel for humidity by repeating the steps with `r._field == "humidity"`.

Save the dashboard with a name like `${USER}-edge-telemetry`.

**Self-check before the checkpoint:**

- [ ] the data source test showed `datasource is working`
- [ ] the temperature panel draws a live line that extends every few seconds
- [ ] the humidity panel does the same
- [ ] the dashboard is saved (it appears under Dashboards)

This is the full local pipeline: data is generated, stored with timestamps, and visualized live, all running in containers on the edge node.

### Checkpoint · Part 9

In [ ]:
import os

grafanaPort = os.environ.get("GRAFANA_PORT", "3000")
grafanaPassword = os.environ.get("GRAFANA_PASSWORD", "edgegrafana12345")
dataSourceQuery = f'curl -s -u admin:{grafanaPassword} http://{deviceAddress()}:{grafanaPort}/api/datasources'
dashboardQuery = f'curl -s -u admin:{grafanaPassword} "http://{deviceAddress()}:{grafanaPort}/api/search?query="'

checkpoint(
    "Part 9 - Grafana dashboard",
    [
        check("Grafana answering", httpOk(f"http://{deviceAddress()}:{grafanaPort}/api/health", expectText="ok"),
              hint="The grafana service is not up - re-run the Part 8 compose cell and its checkpoint first.",
              link="https://grafana.com/docs/grafana/latest/developers/http_api/other/#health-api"),
        check("InfluxDB data source provisioned", outputContains(dataSourceQuery, "influxdb"),
              hint="The provisioning file did not load. Check grafana/provisioning/datasources/influxdb.yaml "
                   "(Part 8 %%writefile cell), then recreate grafana: "
                   "docker compose -p ${USER}-tsdb-lab up -d --force-recreate grafana.",
              link="https://grafana.com/docs/grafana/latest/administration/provisioning/#data-sources"),
        check("a dashboard is saved", outputContains(dashboardQuery, "dash-db"),
              hint="Build and SAVE the dashboard in the Grafana UI (Part 9 steps above) - unsaved panels do "
                   "not count. Name it something like ${USER}-edge-telemetry.",
              link="https://grafana.com/docs/grafana/latest/dashboards/build-dashboards/"),
    ],
    successNote="Grafana is wired to InfluxDB and your live dashboard is saved.",
    docLink="https://grafana.com/docs/grafana/latest/datasources/influxdb/",
)

---
## Part 10 · Create a Retention Bucket

Time-series databases often store high-frequency data. Keeping all raw data forever uses too much storage, so buckets can expire old data automatically.

**[Notebook cell]** With the Compose pipeline still running, list buckets (the original lab does this from another SSH session; `-T` disables the interactive TTY a notebook does not have):

In [ ]:
%cd ~/grafanaTSDBLab/tsdbComposeLab
!docker compose -p ${USER}-tsdb-lab exec -T influxdb \
  influx bucket list \
  --org $INFLUX_ORG \
  --token $INFLUX_TOKEN

**[Notebook cell]** Create a bucket with a 1 hour retention period (re-running reports "bucket already exists" — that is fine):

In [ ]:
!docker compose -p ${USER}-tsdb-lab exec -T influxdb \
  influx bucket create \
  --name shortRetention \
  --org $INFLUX_ORG \
  --retention 1h \
  --token $INFLUX_TOKEN || true

**[Notebook cell]** Write a test point to the short-retention bucket:

In [ ]:
!curl --request POST \
  "http://$DEVICE_ADDR:$INFLUX_PORT/api/v2/write?org=$INFLUX_ORG&bucket=shortRetention&precision=s" \
  --header "Authorization: Token $INFLUX_TOKEN" \
  --header "Content-Type: text/plain; charset=utf-8" \
  --data-binary "device_health,device=sensor01 status=1i"

**[Notebook cell]** Query the short-retention bucket:

In [ ]:
!curl --request POST \
  "http://$DEVICE_ADDR:$INFLUX_PORT/api/v2/query?org=$INFLUX_ORG" \
  --header "Authorization: Token $INFLUX_TOKEN" \
  --header "Accept: application/csv" \
  --header "Content-type: application/vnd.flux" \
  --data 'from(bucket:"short-retention") |> range(start: -10m) |> filter(fn: (r) => r._measurement == "device_health")'

This shows how retention policies control how long time-series data is stored.

### Checkpoint · Part 10

In [ ]:
import os

userName = os.environ.get("USER", "student01")
influxPort = os.environ.get("INFLUX_PORT", "8086")
influxOrg = os.environ.get("INFLUX_ORG", "edge-lab")
influxToken = os.environ.get("INFLUX_TOKEN", "")
bucketListCommand = (
    f'docker exec {userName}-tsdb-lab-influxdb-1 '
    f'influx bucket list --org {influxOrg} --token {influxToken}'
)
healthQuery = (
    f'curl -s --request POST "http://{deviceAddress()}:{influxPort}/api/v2/query?org={influxOrg}" '
    f'--header "Authorization: Token {influxToken}" '
    '--header "Accept: application/csv" --header "Content-type: application/vnd.flux" '
    '--data \'from(bucket:"short-retention") |> range(start: -10m) '
    '|> filter(fn: (r) => r._measurement == "device_health")\''
)

checkpoint(
    "Part 10 - retention bucket",
    [
        check("short-retention bucket exists", outputContains(bucketListCommand, "short-retention"),
              hint="Re-run the `influx bucket create` cell above - the Compose pipeline must be running.",
              link="https://docs.influxdata.com/influxdb/v2/admin/buckets/create-bucket/"),
        check("device_health point stored", outputContains(healthQuery, "device_health"),
              hint="Re-run the write and query cells above. If the point is older than 1 hour it has already "
                   "been expired by retention - write a fresh one.",
              link="https://docs.influxdata.com/influxdb/v2/reference/internals/data-retention/"),
    ],
    successNote="Old data in this bucket now expires automatically after 1 hour - storage stays bounded.",
    docLink="https://docs.influxdata.com/influxdb/v2/admin/buckets/",
)

---
## Part 11 · Simulate Missing Sensor Data

With the full pipeline running, stop only the sensor writer:

In [ ]:
!docker compose -p ${USER}-tsdb-lab stop sensor-writer

Wait about 20 seconds, then look at your Grafana dashboard in the browser. The lines should **flatten or stop**: the gap is visible because no new readings are arriving.

**[Notebook cell]** Query the latest values to confirm the data stopped (the `_time` of the last point stays frozen):

In [ ]:
!curl --request POST \
  "http://$DEVICE_ADDR:$INFLUX_PORT/api/v2/query?org=$INFLUX_ORG" \
  --header "Authorization: Token $INFLUX_TOKEN" \
  --header "Accept: application/csv" \
  --header "Content-type: application/vnd.flux" \
  --data-binary @$HOME/grafanaTSDBLab/queries/latestReading.flux

**[Notebook cell]** Restart the sensor writer:

In [ ]:
!docker compose -p ${USER}-tsdb-lab start sensor-writer

Watch the dashboard recover as new data flows in. This is exactly how an operator notices a device that stopped sending telemetry: the dashboard shows the gap.

### Checkpoint · Part 11

In [ ]:
import os

userName = os.environ.get("USER", "student01")

checkpoint(
    "Part 11 - failure simulated and recovered",
    [
        check("sensorWriter running again", containerRunning(f"{userName}-tsdb-lab-sensor-writer-1"),
              hint="Re-run the `docker compose ... start sensorWriter` cell above to bring the writer back.",
              link="https://docs.docker.com/reference/cli/docker/compose/start/"),
        check("influxdb still up", containerRunning(f"{userName}-tsdb-lab-influxdb-1"),
              hint="Only the writer should have been stopped. If influxdb is down, re-run the Part 8 "
                   "compose cell.",
              link="https://docs.docker.com/compose/"),
    ],
    successNote="You watched telemetry stop and recover - and saw the gap a real operator would see.",
    docLink="https://grafana.com/docs/grafana/latest/dashboards/",
)

---
## Part 12 · Monitor Storage and Stop the Pipeline

Generic container monitoring (`docker stats`, per-service `ps`) works the same here as in the System Architecture lab and the Benchmarking lab. What's specific to a time-series database is how fast it grows on disk.

**[Notebook cell]** Check the database storage size inside the InfluxDB container:

In [ ]:
!docker compose -p ${USER}-tsdb-lab exec -T influxdb du -sh /var/lib/influxdb2

High-frequency sensor data accumulates quickly; this is why the retention bucket in Part 10 matters.

**[Notebook cell]** When finished, stop the pipeline. **Run this only when you are done with Parts 9-11** — the checkpoints above will fail once the stack is down (Cleanup below also runs this, plus volume removal):

In [ ]:
!docker compose -p ${USER}-tsdb-lab down

---
## Cleanup

At the end of the lab, remove your containers, volumes, networks, and images. Only remove your own resources.

**[Notebook cell]** Write a single cleanup script (you can run it here or in a **[Terminal]**):

In [ ]:
%cd ~/grafanaTSDBLab

In [ ]:
%%writefile cleanup.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/grafanaTSDBLab/labEnv.sh

# Compose resources (containers, networks, volumes)
( cd ~/grafanaTSDBLab/tsdbComposeLab 2>/dev/null && docker compose -p "${USER}-tsdb-lab" down -v ) || true

# Standalone containers from Parts 3-6
docker rm -f "${USER}-influxdb" 2>/dev/null || true
docker rm -f "${USER}-sensorWriter" 2>/dev/null || true

# Lab volume, network, and image
docker volume rm "${USER}-influxdbData" 2>/dev/null || true
docker network rm "${USER}-tsdb-net" 2>/dev/null || true
docker rmi "${USER}-sensor-writer" 2>/dev/null || true

echo "Cleanup complete."

In [ ]:
!chmod +x cleanup.sh
!./cleanup.sh

**[Notebook cell]** Optional · remove the whole lab folder (uncomment to run). Only remove your own files, containers, images, networks, and volumes.

In [ ]:
# !rm -rf ~/grafanaTSDBLab

---
### Lab scorecard

In [ ]:
labSummary("Grafana & Time-Series Database Lab")

---
## Part 13 · Connect to Edge Systems

You completed the storage and visualization stages of the edge data pipeline:

```text
Temperature / Environment Sensor
    -> Telemetry Collector         (previous lab)
    -> Time-Series Database        (this lab)
    -> Grafana Dashboard + Alerts  (this lab)
```

A real project could use this stack for equipment monitoring, server health, smart-building sensors, battery and power telemetry, robotics diagnostics, or vehicle telemetry. The main idea: time-series databases are built for timestamped data, and Grafana makes the trends, windows, latest values, and gaps in that data visible at a glance.

---
## Key Terms

- **Time-series database:** a database optimized for many timestamped events (sensor readings, metrics), rather than current state. Here it's InfluxDB.
- **Line protocol:** InfluxDB's text format for a reading `measurement,tag=value field=value`.
- **Measurement / tag / field:** the measurement is the table-like name (`environment`); tags are indexed labels (`device`, `room`); fields are the measured numbers.
- **Flux:** InfluxDB's query language; queries read like a pipeline (`from(...) |> range(...) |> filter(...)`).
- **Bucket:** a named container for data in InfluxDB, optionally with a retention period.
- **Retention policy:** a rule that automatically deletes data older than a set age to control storage use.
- **Grafana:** a dashboarding tool that runs queries against data sources (like InfluxDB) and draws live charts.
- **Data source:** a connection Grafana makes to a backend database so its panels can query it.